# CFD Post-Processing

## Overview
This notebook covers workflows for post-processing CFD results. It includes:
- Loading CFD data (OpenFOAM, Fluent, etc.)
- Computing integrals (drag, lift, heat flux)
- Time series analysis
- Visualization

## Why Post-Processing for Engineering?
- Extract meaningful results from CFD simulations
- Compare with experimental data
- Generate reports and visualizations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import integrate

# 1. Loading CFD Data
print("=== Loading CFD Data ===")

# Simulate CFD time series data
np.random.seed(42)
n_timesteps = 1000
dt = 0.001
t = np.linspace(0, n_timesteps * dt, n_timesteps)

# Force coefficients
cl = 0.5 + 0.1 * np.sin(2 * np.pi * t) + np.random.randn(n_timesteps) * 0.01
cd = 0.02 + 0.005 * np.cos(2 * np.pi * t) + np.random.randn(n_timesteps) * 0.001

# Create a DataFrame
df = pd.DataFrame({
    'time': t,
    'cl': cl,
    'cd': cd
})

print(f"Time series data:")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"Time range: [{t[0]:.3f}, {t[-1]:.3f}] s")

## Computing Integrals

Compute time-averaged quantities and integrals.

In [ ]:
# 2. Computing Integrals
print("\n=== Computing Integrals ===")

# Time-averaged force coefficients
cl_mean = np.mean(cl)
cd_mean = np.mean(cd)
cl_std = np.std(cl)
cd_std = np.std(cd)

print(f"Time-averaged force coefficients:")
print(f"  Cl_mean = {cl_mean:.4f} ± {cl_std:.4f}")
print(f"  Cd_mean = {cd_mean:.4f} ± {cd_std:.4f}")

# Compute lift and drag forces
rho = 1.225  # Air density (kg/m³)
U = 50.0  # Freestream velocity (m/s)
S = 1.0  # Reference area (m²)

q = 0.5 * rho * U**2  # Dynamic pressure
L = cl_mean * q * S
D = cd_mean * q * S

print(f"\nForces:")
print(f"  Lift = {L:.2f} N")
print(f"  Drag = {D:.2f} N")

# Compute integral of a field (e.g., heat flux)
x = np.linspace(0, 1, 100)
heat_flux = 1000 * np.exp(-x**2)  # W/m²
total_heat = integrate.simpson(heat_flux, x=x)

print(f"\nTotal heat flux: {total_heat:.2f} W/m")

## Time Series Analysis

Analyze the time series for periodicity and statistics.

In [ ]:
# 3. Time Series Analysis
print("\n=== Time Series Analysis ===")

# Compute FFT to find dominant frequencies
from scipy.fft import fft, fftfreq

n = len(cl)
fft_values = fft(cl - cl.mean())
freqs = fftfreq(n, d=dt)

# Take only positive frequencies
positive_freqs = freqs[:n//2]
positive_fft = np.abs(fft_values[:n//2])

# Find dominant frequency
dominant_freq = positive_freqs[np.argmax(positive_fft)]
print(f"Dominant frequency: {dominant_freq:.2f} Hz")
print(f"Period: {1/dominant_freq:.3f} s")

# Plot FFT
plt.figure(figsize=(12, 6))
plt.plot(positive_freqs, positive_fft, 'b-')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.title('FFT of Lift Coefficient')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Visualization

Create publication-quality plots.

In [ ]:
# 4. Visualization
print("\n=== Visualization ===")

# Plot time series
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t, cl, 'b-', alpha=0.7)
axes[0].axhline(y=cl_mean, color='r', linestyle='--', label=f'Mean: {cl_mean:.3f}')
axes[0].fill_between(t, cl_mean - cl_std, cl_mean + cl_std, alpha=0.2, color='red', label='±1 Std')
axes[0].set_ylabel('Cl (Lift Coefficient)')
axes[0].set_title('Lift Coefficient Time Series')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, cd, 'r-', alpha=0.7)
axes[1].axhline(y=cd_mean, color='b', linestyle='--', label=f'Mean: {cd_mean:.4f}')
axes[1].fill_between(t, cd_mean - cd_std, cd_mean + cd_std, alpha=0.2, color='blue', label='±1 Std')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Cd (Drag Coefficient)')
axes[1].set_title('Drag Coefficient Time Series')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot histogram of force coefficients
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].hist(cl, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(x=cl_mean, color='r', linestyle='--', label=f'Mean: {cl_mean:.3f}')
axes[0].set_xlabel('Cl')
axes[0].set_ylabel('Count')
axes[0].set_title('Lift Coefficient Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(cd, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(x=cd_mean, color='b', linestyle='--', label=f'Mean: {cd_mean:.4f}')
axes[1].set_xlabel('Cd')
axes[1].set_ylabel('Count')
axes[1].set_title('Drag Coefficient Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercises

1. Load an OpenFOAM time series and compute time-averaged forces.
2. Perform FFT analysis on a CFD time series.
3. Create a publication-quality plot for a report.